# Hito 1 - DocExplore

Este notebook resume la evidencia academica del Hito 1. El objetivo es comparar dos enfoques de clasificacion binaria sobre queries visuales de DocExplore:

- baseline con pixeles redimensionados + flatten + Regresion Logistica
- embeddings CLIP + Regresion Logistica

El alcance sigue excluyendo retrieval, ranking de paginas y localizacion espacial dentro de pagina.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from IPython.display import Image, Markdown, display
import pandas as pd

from src.baseline_pixels import run_baseline_experiment
from src.config import DEFAULT_TARGET_CLASSES, FIGURES_DIR, METRICS_DIR
from src.dataset_inspection import count_valid_page_images, inspect_query_dataset
from src.train_clip_logreg import run_clip_experiment

## 1. Exploracion inicial del dataset

Se reutiliza la inspeccion creada en la fase anterior para contar clases, identificar las mas grandes y dejar trazabilidad de la estructura real del dataset.

In [ ]:
class_counts_path = METRICS_DIR / "class_counts.csv"
if class_counts_path.exists():
    class_counts = pd.read_csv(class_counts_path)
else:
    class_counts = inspect_query_dataset()
    class_counts.to_csv(class_counts_path, index=False)

page_count = count_valid_page_images()
display(class_counts.head(10))
print(f"Cantidad total de clases: {len(class_counts)}")
print(f"Paginas completas validas: {page_count}")

In [ ]:
distribution_path = FIGURES_DIR / "class_distribution.png"
if distribution_path.exists():
    display(Image(filename=str(distribution_path)))
else:
    print(f"No existe aun {distribution_path}. Ejecuta primero python src/dataset_inspection.py")

## 2. Seleccion de clases

El experimento principal del Hito 1 usa por defecto las clases `marqeur` y `simple_sep`, que fueron confirmadas como las mas representativas para partir.

In [ ]:
selected_classes = DEFAULT_TARGET_CLASSES
print(f"Clase 0: {selected_classes[0]}")
print(f"Clase 1: {selected_classes[1]}")

## 3. Baseline con pixeles

Este experimento convierte cada query en una imagen RGB redimensionada, la aplana a un vector de pixeles y entrena una Regresion Logistica con split estratificado.

In [ ]:
baseline_results = run_baseline_experiment(selected_classes=selected_classes)
display(pd.DataFrame([baseline_results["metrics"]]))

In [ ]:
for artifact_name in ("confusion_matrix_png", "threshold_curves_png", "examples_png"):
    artifact_path = baseline_results["artifact_paths"].get(artifact_name)
    if artifact_path:
        display(Markdown(f"**Baseline - {artifact_name}**"))
        display(Image(filename=artifact_path))

## 4. CLIP + Logistic Regression

Aqui se usa `openai/clip-vit-base-patch32` como extractor de embeddings de imagen. Los embeddings se normalizan con norma L2 y luego se entrena una Regresion Logistica binaria.

In [ ]:
clip_results = run_clip_experiment(selected_classes=selected_classes)
display(pd.DataFrame([clip_results["metrics"]]))

In [ ]:
for artifact_name in ("confusion_matrix_png", "threshold_curves_png", "examples_png"):
    artifact_path = clip_results["artifact_paths"].get(artifact_name)
    if artifact_path:
        display(Markdown(f"**CLIP - {artifact_name}**"))
        display(Image(filename=artifact_path))

## 5. Comparacion breve entre enfoques

Se comparan accuracy, precision, recall y F1 para contrastar el baseline con el enfoque basado en embeddings CLIP.

In [ ]:
comparison = pd.DataFrame(
    [
        {"experiment": baseline_results["experiment_name"], **{k: baseline_results["metrics"][k] for k in ["accuracy", "precision", "recall", "f1"]}},
        {"experiment": clip_results["experiment_name"], **{k: clip_results["metrics"][k] for k in ["accuracy", "precision", "recall", "f1"]}},
    ]
).sort_values("f1", ascending=False).reset_index(drop=True)
display(comparison)

## 6. Observaciones finales

- El Hito 1 deja lista una base reproducible para clasificacion binaria sobre queries.
- El analisis por threshold permite discutir el tradeoff precision/recall mas alla del threshold fijo 0.5.
- Los ejemplos TP / FP / TN / FN permiten inspeccion cualitativa de aciertos y errores.
- Retrieval sobre paginas completas y localizacion espacial quedan fuera del alcance de este hito.